# Compare Integrators
---
Last revised by Z. Ellis on 2026 APR 6

## Objectives
This tutorial will demonstrate 

## Imports and Set Up

Here we'll import the necessary libraries and load in the tutorials data folder. Then we define units and frames and load a metakernel to use for time conversions and provide attitude information.

In [1]:
import scarabaeus as scb
from tutorial_data import tutorial_data

import numpy as np
import matplotlib.pyplot as plt

# load tutorial data
data = tutorial_data.load()

## units, frames, kernels
km, hr = scb.Units.get_units(['km', 'hr'])
J2000 = scb.Frame('J2000')

scb.SpiceManager.clear_kernels()    # ensure clean kernel pool
scb.SpiceManager.load_kernel_from_mkfile(data.OREX.mk)
scb.SpiceManager.print_kernels()    # verify C-Kernels for the S/C (sc) and solar arrays (sa)

SCB tutorial data up to date.
                                 Kernels Loaded:
Source:   /Users/zael5647/scarabaeus/docs/online_documentation/sphinx_files/_collections/tutorials/tutorial_data/kernels/scenario/orex_mk.tm   (META)
Source:   tutorial_data/kernels/scenario/orx_sc_rel_210816_210822_v02.bc   (CK)
Source:   tutorial_data/kernels/scenario/orx_sa_rel_210816_210822_v02.bc   (CK)
Source:   tutorial_data/kernels/scenario/ORX_SCLKSCET_00075.tsc   (TEXT)
Source:   tutorial_data/kernels/lsk/naif0012.tls   (TEXT)
Source:   tutorial_data/kernels/scenario/orx_v14.tf   (TEXT)


In [2]:
# set up
km, sec, kg = scb.Units.get_units(['km', 'sec', 'kg'])
J2000 = scb.Frame('J2000')

t0 = scb.SpiceManager.cal2et('2026 JAN 01 00:00:00.000')
tf = scb.SpiceManager.cal2et('2026 JAN 02 00:00:00.000')
epochs = scb.EpochArray(np.arange(t0, tf, 500), 'TDB')

# bodies
jupiter = scb.CelestialBody.from_constants('JUPITER')
sc = scb.Spacecraft('TEST SC', 10000, tot_mass = scb.ArrayWUnits(1e3, kg))

pos0 = scb.ArrayWFrame(scb.ArrayWUnits([25e6, 0, 0], km), J2000)
vel0 = scb.ArrayWFrame(scb.ArrayWUnits([3, 0, 0], km/sec), J2000)

state_vector = scb.StateArray(epoch  = scb.EpochArray(scb.ArrayWUnits(t0, sec), 'TDB'),
                              origin = jupiter,
                              state  = scb.StateDefinition()
                                          .position(sc, pos0)
                                          .velocity(sc, vel0))

# propagator
fm = scb.ForceModelTranslation(sc)
prop = scb.Propagator(primary_body  = sc, 
                      state_vector  = state_vector, 
                      tspan         = epochs, 
                      force_models  = fm, 
                      propagate_STM = False)

# mission sequence
sequence = scb.MissionSequence('TEST_SEQUENCE')
sequence.addLeg('LEG1', scb.ArrayWUnits(86400, sec), state_vector, prop, scb.ArrayWUnits(10, sec))

mission = scb.ScenarioSetup(sequence)
mission.propagate()

# NOTE: cant specify file location so this will fail when it tries to write the file
orbiter_traj = scb.Trajectory('TEST_TRAJ', state_array = mission.propagated_state_array)

/Users/zael5647/scarabaeus/src/scarabaeus/utils/SCBPolynomial.py:80: RankWarning: Polyfit may be poorly conditioned
  coeffs = np.polyfit(t_array, y_array, deg)



--- Propagating Segment 1/1: 'LEG1' [Leg] ---

[IC] segment='LEG1' type='Leg' epoch=820497669.18392 sec (TDB)
  position     n= 3 sid=10000 frame=J2000 vals[:6]=[25000000.        0.        0.]
  velocity     n= 3 sid=10000 frame=J2000 vals[:6]=[3. 0. 0.]

                            Starting propagation...                             


Integrating:   0%|                                                       | 0.00/86400.00 s [00:00<?]

Integrating:   0%|                                                       | 0.00/86400.00 s [00:00<?]

Integrating:   0%|                                                   | 0.10/86400.00 s [00:00<33:05]

...

Integrating: 100%|███████████████████████████████████████████████| 86400.00/86400.00 s [00:00<00:00]

Integrating: 100%|███████████████████████████████████████████████| 86400.00/86400.00 s [00:00<00:00]

Integrating: 100%|███████████████████████████████████████████████| 86400.00/86400.00 s [00:00<00:00]

Integrating: 100%|███████████████████████████████████████████████| 86400.00/86400.00 s [00:00<00:00]


 =================== DOP853 integration complete. ==================
Propagation complete.

[GLOBAL STATE after 'LEG1']
  key=('position', 3, 10000, 'J2000', 0) -> global[0:3] = [25258448.613674797        0.                 0.         ]
  key=('velocity', 3, 10000, 'J2000', 0) -> global[3:6] = [2.9826662229935 0.              0.             ]
